In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
import scipy as sp
import seaborn as sns
from ising.model import FitMethod

from climate_attitudes.correlation import Correlation
from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets.common import IndexColumn
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import plot_corr_network
from ising import Ising

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202605211526

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced_no_imputation", with_imputation=False)
pids, Y0, _ = dataset.indices_to_numpy(
    kind="time-series", binarise=False, seed=RANDOM_SEED
)
_, Y, X = dataset.indices_to_numpy(
    kind="time-series", binarise=True, scale=0.25, seed=RANDOM_SEED
)
node_labels = np.asarray(dataset.schema.get_short_names(kind="measurement"))
indices = dataset.indices.collect()

In [ ]:
index_cols = [
    col for col in dataset.schema.post_index().columns if isinstance(col, IndexColumn)
]
index_cols

In [ ]:
dataset.schema.post_index().columns[-1]

In [ ]:
dataset.indices.collect().sort(by=("participant_id", "wave"))

In [ ]:
dataset.response.collect().sort(by=("participant_id", "wave"))

In [ ]:
x = dataset.indices.collect().sort(by=("participant_id", "wave")).select("politics")
y = (
    dataset.response.collect()
    .sort(by=("participant_id", "wave"))
    .select(pl.mean_horizontal("pol_affiliation", "pol_ideology"))
)

plt.scatter(x, y)

In [ ]:
indices.sort(by=("participant_id", "wave")).select(
    *dataset.schema.get_cols("measurement")
).to_numpy().ravel().reshape((-1, 2, len(dataset.schema.get_cols("measurement"))))

In [ ]:
Y

Isolate first wave only

In [ ]:
Y0 = Y0[:, 0]

Compute pairwise correlations between rows

In [ ]:
rowcorrs = np.corrcoef(Y0)
rowcorrs[abs(rowcorrs) < 0.05] = 0.0
rowcorrs[np.diag_indices_from(rowcorrs)] = 0.0
# rowcorrs[np.tril_indices_from(rowcorrs)] = 0.0

Partition network into modules

In [ ]:
def leading_eig_comm_recursive(adj, levels, parent_label):
    if levels == 0:
        return np.full(adj.shape[0], fill_value=parent_label, dtype=np.int64)

    G = nx.from_numpy_array(adj)
    mod = nx.modularity_matrix(G)
    eigs, eigv = sp.linalg.eig(mod)

    if parent_label == 1:
        neg_label, pos_label = 2, 3
    else:
        neg_label, pos_label = parent_label * 2, parent_label * 2 + 1

    labels = np.where(eigv[0] >= 0, pos_label, neg_label)
    is_neg_label = labels == neg_label
    is_pos_label = labels == pos_label

    labels[is_neg_label] = leading_eig_comm_recursive(
        adj[is_neg_label][:, is_neg_label], levels - 1, neg_label
    )
    labels[is_pos_label] = leading_eig_comm_recursive(
        adj[is_pos_label][:, is_pos_label], levels - 1, pos_label
    )

    return labels


def leading_eig_comm(adj, levels):
    labels = leading_eig_comm_recursive(adj, levels, 1)
    for i, label in enumerate(range(2**levels, 2 ** (levels + 1))):
        labels[labels == label] = i
    return labels

In [ ]:
levels = 1
comms = leading_eig_comm(rowcorrs, levels)

Plot partial correlations from each group

In [ ]:
plot_corr_network(
    dataset.indices.drop("participant_id", "wave", pl.col(r"^dem_.*$")).collect(),
    mask_below=0.1,
    kind=Correlation.PARTIAL,
)

In [ ]:
for i in range(2**levels):
    subset_pids = pids[comms == i]
    subset_ind = dataset.indices.filter(pl.col("participant_id").is_in(subset_pids))
    plot_corr_network(
        subset_ind.drop("participant_id", "wave", pl.col(r"^dem_.*$")).collect(),
        mask_below=0.1,
        kind=Correlation.PARTIAL,
    )

In [ ]:
G = nx.from_numpy_array(rowcorrs)
mod = nx.modularity_matrix(G)

In [ ]:
eigs, eigv = sp.linalg.eig(mod)
np.where(eigv[0] >= 0, 1, -1)

In [ ]:
model = Ising.fit(
    Y,
    X=X,
    method=FitMethod.TIME_SERIES,
    node_labels=node_labels,
    rng=RANDOM_SEED,
    self_loops=True,
)

In [ ]:
init_energy = np.empty(Y.shape[0], dtype=np.float64)
final_energy = np.empty(Y.shape[0], dtype=np.float64)

spins_changed = np.empty(Y.shape[0], dtype=np.int64)
for i in range(Y.shape[0]):
    init_energy[i] = -(
        model.parallel_glauber_theta(
            Y[i, 0], np.concat(([1], X[i, 0])), model.h, model.j, model.adj
        )
        @ Y[i, 0]
    )
    final_energy[i] = -(
        model.parallel_glauber_theta(
            Y[i, 1], np.concat(([1], X[i, 1])), model.h, model.j, model.adj
        )
        @ Y[i, 1]
    )
    spins_changed[i] = (np.abs(Y[i, 1] - Y[i, 0]) // 2).sum()

energy_change = final_energy - init_energy

In [ ]:
model.node_labels

In [ ]:
Y[1525, 0]

In [ ]:
Y[1525, 1]

In [ ]:
Y0[1525, 0]

In [ ]:
Y0[1525, 1]

In [ ]:
Y[-2, 0]

In [ ]:
init_energy.argmax()

In [ ]:
sns.relplot(x=init_energy, y=spins_changed)

In [ ]:
sns.relplot(x=init_energy, y=final_energy)

In [ ]:
np.concat(([1], X[0, 0]))